# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya – Exploration with `mlcroissant`

This notebook demonstrates how to load, examine, and process the FAIR^2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, following the Croissant specification. All dataset entities—including record sets, fields, and columns—are referenced by their `@id` for transparency and reproducibility.

### Dataset Source
The Croissant schema for this dataset is located at:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Install mlcroissant if it is not installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and examine its main features.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# For pretty printing metadata fields
metadata_json = dataset.metadata.to_json()

print("Dataset Name:", metadata_json.get('name', ''))
print("Dataset Description:", metadata_json.get('description', ''))
print("Cite As:", metadata_json.get('citeAs', ''))
print("License:", metadata_json.get('license', ''))
print("Temporal Coverage:", metadata_json.get('temporalCoverage', ''))
print("Spatial Coverage:", metadata_json.get('spatialCoverage', ''))
print("Keywords:", metadata_json.get('keywords', ''))

## 2. Data Overview

Let's get an overview of available record sets in the dataset, as well as their fields. Entities are referenced by their `@id`.

> **Note:** If you are unfamiliar with Croissant structure, a **record set** corresponds to a logical table, with **fields** describing the columns. The `mlcroissant` library exposes these via `dataset.record_sets()`.

In [ ]:
# List all record sets and their IDs
record_sets = list(dataset.record_sets())

if record_sets:
    print("Record sets found in dataset:")
    for rs in record_sets:
        print(f"@id: {rs['@id']}")
        print(f"  name: {rs.get('name','')}")
        print(f"  description: {rs.get('description','')}")
        # List fields for each record set
        if 'fields' in rs and rs['fields']:
            print("  Fields:")
            for field in rs['fields']:
                print(f"    - @id: {field['@id']}, name: {field.get('name','')}, dataType: {field.get('dataType','')}")
        print()
else:
    print("No record sets detected in this dataset.\nTrying to list available distributions (data files) as fallback...")
    for dist in dataset.metadata.to_json().get('distribution', []):
        print(json.dumps(dist, indent=2))

## 3. Data Extraction

Load data from specific record sets (`@id` references) into pandas DataFrames. The record sets and fields are identified using their `@id` as listed in the previous step.

> If there are no traditional record sets, but data files are defined as `distribution`, we try to load the records using their file `@id`.

In [ ]:
# If there are record sets, load the data from each. Otherwise, try the dataset distributions.
dataframes = {}

if record_sets:
    for rs in record_sets:
        rs_id = rs['@id']
        print(f"\nLoading records from record set @id = {rs_id}")
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records. Columns:", df.columns.tolist())
            display(df.head(3))
        except Exception as e:
            print(f"Could not load records for record set {rs_id}: {e}")
else:
    # Fallback for Croissant datasets that are file-centric and do not define record sets
    print("Loading records from distributions (@id of files)...")
    for dist in dataset.metadata.to_json().get('distribution', []):
        dist_id = dist['@id'] if isinstance(dist, dict) else dist
        print(f"Trying distribution @id = {dist_id}")
        try:
            records = list(dataset.records(record_set=dist_id))
            df = pd.DataFrame(records)
            dataframes[dist_id] = df
            print(f"Loaded {len(df)} records. Columns:", df.columns.tolist())
            display(df.head(3))
        except Exception as e:
            print(f"Could not load records from distribution {dist_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field from one of the available record sets (by `@id`) for simple EDA: filtering, normalization, and grouping.

> *Adjust the selected record set `@id` and field `@id`/name below if necessary, according to what was found above.*

In [ ]:
import numpy as np

# Choose a record set and a numeric field for EDA
# Update these values according to the output above
if dataframes:
    # Select first record set with non-empty DataFrame
    for record_set_id, df in dataframes.items():
        if not df.empty:
            break
    print(f"\nProceeding with record set @id: {record_set_id}")
    print(df.head())
    print("Available columns:", df.columns.tolist())
    
    # Try to auto-select a numeric field (float or int type)
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dropna().dtype, np.number):
            numeric_field = col
            break
    
    if numeric_field is None:
        # If no numeric field, ask user to select; fallback to first string column
        numeric_field = df.columns[0]
        print(f"No numeric columns detected; selecting the first column: {numeric_field}")
    else:
        print(f"Chosen numeric field for EDA: {numeric_field}")
    
    threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else None
    if threshold is not None:
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where '{numeric_field}' > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalizing the chosen numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nAdd normalized column '{numeric_field}_normalized':")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by a suitable (likely categorical) field
        group_field = None
        for col in df.columns:
            if col != numeric_field and (df[col].dtype == 'object' or str(df[col].dtype).startswith('category')):
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            grouped_df.columns = [f"mean_{numeric_field}"]
            print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No categorical/grouping field found for this record set.")
    else:
        print("No numeric threshold possible (field not numeric). Skipping filtering/normalization.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and, if available, relationships with a grouping/categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field in df.columns:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True, color='steelblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # Boxplot grouped by categorical field if available
    if 'group_field' in locals() and group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:

- Load a dataset defined with a Croissant schema using the `mlcroissant` library, referencing entities by their `@id` for reproducible data science.
- Explore and inspect record sets, fields, and data content.
- Extract tables into pandas DataFrames for downstream processing.
- Apply simple exploratory analysis (filtering, normalization, grouping) and visualize major patterns.

This approach enables transparent, schema-aware, and FAIR data workflows. You can further extend this analysis using the available `@id` references to slice, merge, or join data based on the detailed Croissant definitions.